# 03 - Production monitoring (simulated traffic)

**This is a simulation.** `src/simulate_traffic.py` sent 30 days x 2,000 requests, sampled
from the held-out TEST set, through the real API. Nothing was ever visited or resolved:
hostnames are just text.

* **Days 1-10 - normal:** the test set's natural mix (~21% phishing).
* **Days 11-20 - free-hosting campaign:** 35% phishing, most of the extra from free-hosting sites.
* **Days 21-30 - clean-looking phishing:** 25% phishing, only hostnames with no subdomain,
  digits, hyphens, risky TLD or free hosting.

The **reference** is a 20% random holdout of the test set that the simulation never used.
Labels (`logs/simulated_labels.csv`) are only used for the "realized" curves; every other
check works without them.

*Why not NannyML?* A dry-run install shows `nannyml` would pull in about 40 extra packages
(cloud-storage clients, lightgbm, FLAML...) and its 0.13 line predates pandas 3, which this
project uses. CBPE is a few lines of numpy, so `src/monitor.py` implements it directly.

In [1]:
import os, sys
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
import pandas as pd
from IPython.display import display
from src import monitor as m
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)

daily, scores, thresholds, limits, alerts, info = m.run_monitoring()
print(f"threshold {info['threshold']:.4f} | reference rows {info['ref_rows']} "
      f"({info['ref_phishing_share']:.1%} phishing) | most features drifting on any "
      f"normal reference slice: {int(info['null_counts'].max())}")

2026/10/08 19:02:32 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Phishing_URL_Project\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


C:\Phishing_URL_Project\venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


threshold 0.6260 | reference rows 16276 (20.7% phishing) | most features drifting on any normal reference slice: 0


## Limits used (all derived from the reference)

* **Feature drift:** KS statistic (continuous) or difference in proportions (binary). A
  feature drifts when its score beats its own threshold = max(0.05, 1.1 x the 99th
  percentile of the score on 200 random 2,000-row slices of the reference).
* **Everything else** (decision shares, mean probability, estimated and realized precision
  and recall): alert outside *reference mean +/- 3 standard deviations*, where the standard
  deviation comes from the same 200 slices.
* **Drifting-feature count:** alert when more than 2 features drift on the same day.

In [2]:
display(limits.round(4)); display(thresholds.round(3).to_frame('feature_threshold').T)

,low,high,reference
share_safe,0.8474,0.8878,0.8676
share_phishing,0.0490,0.0787,0.0639
share_review,0.0523,0.0849,0.0686
mean_probability,0.1903,0.2287,0.2095
est_precision,0.9122,0.9466,0.9294
est_recall,0.5410,0.6335,0.5873
real_precision,0.8662,0.9616,0.9139
real_recall,0.5150,0.6504,0.5827


,hostname_length,num_labels,num_subdomain_levels,num_digits,digit_ratio,num_hyphens,longest_label_length,domain_entropy,vowel_ratio,is_ip,...,suspicious_keyword_count,is_free_hosting,tld_risky,name_length,name_entropy,name_digit_ratio,name_num_hyphens,name_keyword_count,name_brand_lookalike,name_homoglyph_brand
feature_threshold,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,...,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.05


## 1. Data drift - how many features moved

Each bar is one day. A feature is counted when its distribution that day is further from
the reference than normal day-to-day noise would explain. Taller bars = the incoming
hostnames look less like what the model was tested on. The dashed line is the alert limit.

In [3]:
m.chart_drifting_features(daily); None

The heat map shows **which** features moved: bright (above 1) means that feature drifted
on that day. It lets you see *what kind* of change happened, not just that something did.

In [4]:
m.chart_drift_heatmap(scores, thresholds); None

## 2. Prediction drift - what the model decides

Top: the share of requests the model calls *safe*, *phishing* or *review* (free-hosting
sites above the threshold). The shaded bands are the normal range. Bottom: the average
probability the model outputs. If the world changes, these usually move first, and they
need no labels.

In [5]:
m.chart_decision_shares(daily, limits); None

## 3. Estimated vs. realized performance

The **blue line** is the performance the model *thinks* it has, estimated from its own
probabilities without any labels (CBPE: if the model says 0.9, count it as 0.9 of a true
phishing page and 0.1 of a false alarm, then add up). The **orange line** is the real
performance, computed afterwards from the true labels. The blue band is the 95% range of
the estimate; the grey band is the normal range. Where the lines agree, you can trust the
label-free estimate. Where they split apart, the model is confidently wrong in a way its
own probabilities cannot reveal.

In [6]:
m.chart_estimated_vs_realized(daily, limits); None

## 4. Alert table (day, check, value, limit)

In [7]:
display(alerts.round(4).reset_index(drop=True))

,day,check,value,limit,direction
0,7,est_recall,0.5404,0.5410,below
1,11,drifting_features,8.0000,2.0000,above
2,11,share_safe,0.7510,0.8474,below
3,11,share_review,0.1920,0.0849,above
4,11,mean_probability,0.3174,0.2287,above
...,...,...,...,...,...
150,30,mean_probability,0.1628,0.1903,below
151,30,est_precision,0.9000,0.9122,below
152,30,est_recall,0.3013,0.5410,below
153,30,real_precision,0.8624,0.8662,below


## 5. Summary per scenario

Average of the daily values, plus how many alerts fired. Note how the alerts line up with
the scenarios, and which checks stay quiet in the last one.

In [8]:
daily["scenario"] = pd.cut(daily.index, [0, 10, 20, 30],
                           labels=["days 1-10 normal", "days 11-20 free-hosting",
                                   "days 21-30 clean-looking"])
cols = ["true_phishing_share", "n_drifting", "share_safe", "share_phishing", "share_review",
        "mean_probability", "est_precision", "real_precision", "est_recall", "real_recall"]
summary = daily.groupby("scenario", observed=True)[cols].mean()
a = alerts.assign(scenario=pd.cut(alerts["day"], [0, 10, 20, 30], labels=summary.index))
summary["alerts"] = a.groupby("scenario", observed=True).size().reindex(summary.index).fillna(0).astype(int)
display(summary.round(3))
display(a.groupby(["scenario", "check"], observed=True).size().unstack(0).fillna(0).astype(int))

,true_phishing_share,n_drifting,share_safe,share_phishing,share_review,mean_probability,est_precision,real_precision,est_recall,real_recall,alerts
scenario,,,,,,,,,,,
days 1-10 normal,0.21,0.0,0.870,0.061,0.069,0.208,0.932,0.929,0.581,0.575,1
days 11-20 free-hosting,0.35,10.1,0.744,0.064,0.191,0.323,0.951,0.975,0.754,0.712,78
days 21-30 clean-looking,0.25,10.5,0.943,0.057,0.000,0.162,0.886,0.851,0.311,0.194,76


scenario,days 1-10 normal,days 11-20 free-hosting,days 21-30 clean-looking
check,,,
drifting_features,0,10,10
est_precision,0,9,10
est_recall,1,10,10
mean_probability,0,10,10
real_precision,0,9,6
real_recall,0,10,10
share_review,0,10,10
share_safe,0,10,10


## Reading the result

* **Normal days (1-10):** no drift and no meaningful alert; estimated and realized lines
  overlap (recall 0.58 vs 0.58, precision 0.93 vs 0.93).
* **Free-hosting campaign (11-20):** data drift (about 10 features, incl. `is_free_hosting`,
  digits, entropy, hyphens) and prediction drift (the *review* share more than doubles).
  The label-free estimate follows reality in direction but is a bit optimistic on recall
  (about +0.04) and slightly pessimistic on precision.
* **Clean-looking phishing (21-30):** every check fires. Here the drift is partly a side
  effect of how the scenario is built: all phishing with subdomains, digits, hyphens or free
  hosting is missing, so the incoming mix looks very different from the reference. The model
  also misses most of the remaining phishing. CBPE sees the drop (estimated recall
  0.31, down from 0.58) but **under-states** it: realized recall is 0.19. The estimate is
  only as good as the probabilities, and the model is over-confident on exactly this kind of
  hostname. Treat the estimate as an early warning, not as the true number.